In [1]:
import re
import urllib
import ssl
import urllib.request 
from time import sleep
import bs4
import requests
import pandas as pd
#from TableParser import TableParser

features = ['rank','name','position','team','games','minutes','off_rpm','def_rpm','rpm','wins','year']
#headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:63.0) Gecko/20100101 Firefox/63.0'}
headers={'User-Agent': 'Mozilla/5.0'}
#headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
# 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
# 'Accept-Language': 'en-US,en;q=0.9',
# 'Accept-Encoding': 'gzip, deflate, br'}
#headers = {
#    'accept': '*/*',
#    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.64 Safari/537.36 Edg/101.0.1210.53',
#    'Accept-Language': 'en-US,en;q=0.9,it;q=0.8,es;q=0.7',
#    'referer': 'https://www.google.com/',
#    'cookie': 'DSID=AAO-7r4OSkS76zbHUkiOpnI0kk-X19BLDFF53G8gbnd21VZV2iehu-w_2v14cxvRvrkd_NjIdBWX7wUiQ66f-D8kOkTKD1BhLVlqrFAaqDP3LodRK2I0NfrObmhV9HsedGE7-mQeJpwJifSxdchqf524IMh9piBflGqP0Lg0_xjGmLKEQ0F4Na6THgC06VhtUG5infEdqMQ9otlJENe3PmOQTC_UeTH5DnENYwWC8KXs-M4fWmDADmG414V0_X0TfjrYu01nDH2Dcf3TIOFbRDb993g8nOCswLMi92LwjoqhYnFdf1jzgK0'
#}
context = ssl._create_unverified_context()

def process_year_page(oddrows,evenrows,year):
    allrows = oddrows + evenrows
    result = []
    for row in allrows:
        result.append([])
        allcols = row.findAll('td')
        for col in allcols:
          thestrings = [s for s in col.findAll(string=True)]
          thetext = ''.join(thestrings)
          result[-1].append(thetext)

    result
    df = pd.DataFrame(result, columns = ['rank','name, position','team','games','minutes','off_rpm','def_rpm','rpm','wins'])
    df['name'] = df['name, position'].apply(lambda x: x.split(',')[0])
    df['position'] = df['name, position'].apply(lambda x: x.split(', ')[1])
    df['year'] = year
    return df[features]



years = list(range(2024, 2025))
#https://www.espn.com/nba/statistics/rpm/_/year/2023/page/2

df = pd.DataFrame(columns = features)
for year in years:
    first_page = 'https://www.espn.com/nba/statistics/rpm/_/year/' + str(year)
    # here we define the headers for the request
    req = urllib.request.Request(url=first_page, headers=headers)
    f = urllib.request.urlopen(req,context=context)
    page_source = f.read().decode('utf-8')
    soup = bs4.BeautifulSoup(page_source)
    page_numbers = soup.find_all("div", {'class':'page-numbers'})
    page_numbers_text = str(page_numbers[0].findAll(string=True)).replace("]", "").replace("[", "").replace("'", "")
    print(page_numbers_text)
    
    max_pages = int(page_numbers_text.split('of ')[1])
    print(max_pages)
    oddrows = soup.find_all("tr", {'class':'oddrow'})
    evenrows = soup.find_all("tr", {'class':'evenrow'})
    df_new = process_year_page(oddrows,evenrows,year)
    df = pd.concat([df,df_new])
    df.to_csv('rpm_database_'+str(year)+'.csv',index = False)    
    sleep(10)
    for page in range(2,(max_pages + 1)):
        next_page = 'https://www.espn.com/nba/statistics/rpm/_/year/' + str(year) + '/page/' + str(page)
        context = ssl._create_unverified_context()
        req = urllib.request.Request(url=next_page, headers=headers)
        try:
            f = urllib.request.urlopen(req,context=context)
            page_source = f.read().decode('utf-8')
            soup = bs4.BeautifulSoup(page_source)
            oddrows = soup.find_all("tr", {'class':'oddrow'})
            evenrows = soup.find_all("tr", {'class':'evenrow'})
            df_new = process_year_page(oddrows,evenrows,year)
            df = pd.concat([df,df_new])
            df.to_csv('rpm_database_'+str(year)+'.csv',index = False)    
            print('yay')
            
        except:
            try:
                headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
                     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
                     'Accept-Language': 'en-US,en;q=0.9',
                     'Accept-Encoding': 'gzip, deflate, br'}
                req = urllib.request.Request(url=next_page, headers=headers)
                context = ssl._create_unverified_context()
                f = urllib.request.urlopen(req,context=context)    
                page_source = f.read().decode('utf-8')
                soup = bs4.BeautifulSoup(page_source)
                oddrows = soup.find_all("tr", {'class':'oddrow'})
                evenrows = soup.find_all("tr", {'class':'evenrow'})
                df_new = process_year_page(oddrows,evenrows,year)
                df = pd.concat([df,df_new])
                df.to_csv('rpm_database_'+str(year)+'.csv',index = False)
                print('oops')
                
            except:
                headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:63.0) Gecko/20100101 Firefox/63.0'}
                req = urllib.request.Request(url=next_page, headers=headers)
                context = ssl._create_unverified_context()
                f = urllib.request.urlopen(req,context=context)
                page_source = f.read().decode('utf-8')
                soup = bs4.BeautifulSoup(page_source)
                oddrows = soup.find_all("tr", {'class':'oddrow'})
                evenrows = soup.find_all("tr", {'class':'evenrow'})
                df_new = process_year_page(oddrows,evenrows,year)
                df = pd.concat([df,df_new])
                df.to_csv('rpm_database_'+str(year)+'.csv',index = False)
                print('oops')
            
        #page_source = f.read().decode('utf-8')
        #soup = bs4.BeautifulSoup(page_source)
        #oddrows = soup.find_all("tr", {'class':'oddrow'})
        #evenrows = soup.find_all("tr", {'class':'evenrow'})
        #df_new = process_year_page(oddrows,evenrows,year)
        #df = pd.concat([df,df_new])
        #df.to_csv('rpm_database_'+str(year)+'.csv',index = False)    
        print(page)
        sleep(10)
    print(year)
        
#df.to_csv('rpm_database.csv',index = False)        
    
    
#print(divs)
#for div in divs:
#    print(div)

HTTPError: HTTP Error 404: Not Found

In [ ]:
def process_rows(oddrows,evenrows,year):
    features = ['rank','name','position','team','games','minutes','off_rpm','def_rpm','rpm','wins','year']
    allrows = oddrows + evenrows
    result = []
    for row in allrows:
        result.append([])
        allcols = row.findAll('td')
        for col in allcols:
          thestrings = [s for s in col.findAll(string=True)]
          thetext = ''.join(thestrings)
          result[-1].append(thetext)

    result
    df = pd.DataFrame(result, columns = ['rank','name, position','team','games','minutes','off_rpm','def_rpm','rpm','wins'])
    df['name'] = df['name, position'].apply(lambda x: x.split(',')[0])
    df['position'] = df['name, position'].apply(lambda x: x.split(', ')[1])
    df['year'] = year
    return df[features]

In [ ]:
oddrows + evenrows

In [ ]:
import requests
import pandas as pd

url = 'https://www.espn.com/nba/statistics/rpm'
html = requests.get(url).content
df_list = pd.read_html(html)
#teams_source = f.read().decode('utf-8')

#url = urlopen(req)
#tp = TableParser()
#tp.feed(teams_source)

# NOTE: Here you need to know exactly how many tables are on the page and which one
# you want. Let's say it's the first table
#my_table = tp.get_tables()[0]
#filename = 'table_as_csv.csv'
#f = open(filename, 'wb')
#with f:
#    writer = csv.writer(f)
#    for row in table:
#        writer.writerow(row

In [ ]:
from urllib import Request, urlopen, URLError
from TableParser import TableParser
url_addr ='https://www.espn.com/nba/statistics/rpm'
req = Request(url_addr)
url = urlopen(req)
tp = TableParser()
tp.feed(url.read())

# NOTE: Here you need to know exactly how many tables are on the page and which one
# you want. Let's say it's the first table
my_table = tp.get_tables()[0]
filename = 'table_as_csv.csv'
f = open(filename, 'wb')
with f:
    writer = csv.writer(f)
    for row in table:
        writer.writerow(row)

In [ ]:
pip install html5lib